# 介紹
本程式使用 pyautogui 來完成一個 Blitz 的遊戲機器人。

In [32]:
import pyautogui as pyg
pyg.sleep(2)
basex=157
basey=385
gap=50

In [33]:
for row in range(8):
    for col in range(8):
        x = basex + col * gap
        y = basey + row * gap
        color = pyg.pixel(x, y)
        print(row, col, x, y, color)

0 0 157 385 (128, 75, 119)
0 1 207 385 (150, 95, 145)
0 2 257 385 (177, 110, 134)
0 3 307 385 (162, 102, 129)
0 4 357 385 (149, 90, 165)
0 5 407 385 (20, 7, 29)
0 6 457 385 (29, 17, 40)
0 7 507 385 (20, 7, 29)
1 0 157 435 (200, 183, 159)
1 1 207 435 (200, 183, 159)
1 2 257 435 (221, 152, 150)
1 3 307 435 (216, 150, 147)
1 4 357 435 (142, 86, 159)
1 5 407 435 (35, 16, 46)
1 6 457 435 (25, 7, 33)
1 7 507 435 (35, 17, 47)
2 0 157 485 (26, 120, 178)
2 1 207 485 (193, 173, 142)
2 2 257 485 (244, 184, 170)
2 3 307 485 (242, 180, 173)
2 4 357 485 (134, 82, 153)
2 5 407 485 (31, 10, 40)
2 6 457 485 (44, 20, 54)
2 7 507 485 (31, 10, 40)
3 0 157 535 (163, 139, 97)
3 1 207 535 (163, 139, 97)
3 2 257 535 (249, 190, 191)
3 3 307 535 (249, 195, 180)
3 4 357 535 (126, 76, 147)
3 5 407 535 (58, 29, 63)
3 6 457 535 (43, 18, 48)
3 7 507 535 (58, 29, 63)
4 0 157 585 (140, 108, 87)
4 1 207 585 (129, 102, 78)
4 2 257 585 (252, 199, 200)
4 3 307 585 (252, 199, 195)
4 4 357 585 (120, 72, 142)
4 5 407 585 (58

In [34]:

def get_gem_position(r, c):
    x = basex + c * gap
    y = basey + r * gap
    return (x, y)

In [35]:

def get_gem_color(r, c, im):
    x, y = get_gem_position(r, c)
    return im.getpixel((x, y))

In [36]:
def can_move_left(r, c, im):
    if not (0 <= r < 8 and 1 <= c < 8):
        return False

    left_c = c - 1
    left_color = get_gem_color(r, left_c, im)
    current_color = get_gem_color(r, c, im)

    def color_after_swap(row, col):
        if (row, col) == (r, left_c):
            return current_color
        if (row, col) == (r, c):
            return left_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for dr, dc in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                nr = row + dr * direction
                nc = col + dc * direction
                while (0 <= nr < 8 and 0 <= nc < 8
                       and color_after_swap(nr, nc) == color):
                    count += 1
                    nr += dr * direction
                    nc += dc * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, left_c) or makes_match(r, c)

In [37]:

def _can_swap_and_match(r, c, dr, dc, im):
    nr = r + dr
    nc = c + dc
    if not (0 <= r < 8 and 0 <= c < 8 and 0 <= nr < 8 and 0 <= nc < 8):
        return False

    current_color = get_gem_color(r, c, im)
    neighbor_color = get_gem_color(nr, nc, im)

    def color_after_swap(row, col):
        if (row, col) == (r, c):
            return neighbor_color
        if (row, col) == (nr, nc):
            return current_color
        return get_gem_color(row, col, im)

    def makes_match(row, col):
        color = color_after_swap(row, col)
        for axis_r, axis_c in ((0, 1), (1, 0)):
            count = 1
            for direction in (-1, 1):
                check_r = row + axis_r * direction
                check_c = col + axis_c * direction
                while (0 <= check_r < 8 and 0 <= check_c < 8
                       and color_after_swap(check_r, check_c) == color):
                    count += 1
                    check_r += axis_r * direction
                    check_c += axis_c * direction
            if count >= 3:
                return True
        return False

    return makes_match(r, c) or makes_match(nr, nc)

def can_move_right(r, c, im):
    return _can_swap_and_match(r, c, 0, 1, im)

def can_move_up(r, c, im):
    return _can_swap_and_match(r, c, -1, 0, im)

def can_move_down(r, c, im):
    return _can_swap_and_match(r, c, 1, 0, im)

In [38]:
im = pyg.screenshot()
print(can_move_left(5, 4, im))

False


In [39]:

print(get_gem_position(0,0))
print(get_gem_position(0,1))
print(get_gem_position(1,0))

(157, 385)
(207, 385)
(157, 435)


In [40]:
im = pyg.screenshot()
for i in range(8):
    x, y = get_gem_position(0, i)
    pyg.click(x, y, duration=1)
    print(get_gem_color(0, i, im))

(128, 75, 119)
(150, 95, 145)
(177, 110, 134)
(162, 102, 129)
(149, 90, 165)
(20, 7, 29)
(29, 17, 40)
(20, 7, 29)


In [41]:
import time

start_time = time.time()
im = pyg.screenshot()

while True:
    if time.time() - start_time > 60:
        break

    for r in range(8):
        for c in range(8):
            if time.time() - start_time > 60:
                break

            directions = (
                (can_move_left, 0, -1),
                (can_move_right, 0, 1),
                (can_move_up, -1, 0),
                (can_move_down, 1, 0),
            )
            for can_move, dr, dc in directions:
                if can_move(r, c, im):
                    x1, y1 = get_gem_position(r, c)
                    x2, y2 = get_gem_position(r + dr, c + dc)
                    pyg.moveTo(x1, y1)
                    pyg.dragTo(x2, y2, duration=0.2)
                    pyg.sleep(0.2)
                    im = pyg.screenshot()
                    break

        if time.time() - start_time > 60:
            break

KeyboardInterrupt: 

In [ ]:
import time
import pyautogui as pyg

# 1. 記錄目前的時間
start_time = time.time()

# 2. 初始螢幕截圖
im = pyg.screenshot() 

print("開始自動遊戲機制...")

# 3. 進入無窮迴圈
while True:
    # 檢查是否已超過 1 分鐘 (60 秒)
    if time.time() - start_time > 60:
        print("已執行超過 1 分鐘，跳出無窮迴圈。")
        break
def move_left(r, c):
    # 請在此處寫上向左移動寶石的程式碼
    pass

def move_right(r, c):
    # 請在此處寫上向右移動寶石的程式碼
    pass

def move_up(r, c):
    # 請在此處寫上向上移動寶石的程式碼
    pass

def move_down(r, c):
    # 請在此處寫上向下移動寶石的程式碼
    pass
    # 假設棋盤為 8x8，請依實際遊戲棋盤大小調整 range
    # 遍歷畫面中的每顆寶石 (r, c)
    moved_in_this_pass = False
    for r in range(8):
        for c in range(8):
            # 判斷四個方向是否可移動 (參數順序傳入 r, c, im)
            if can_move_left(r, c, im):
                move_left(r, c)       # 執行向左移動
                im = pyg.screenshot() # 移動後重新截圖
                moved_in_this_pass = True
                break
            elif can_move_right(r, c, im):
                move_right(r, c)      # 執行向右移動
                im = pyg.screenshot() # 移動後重新截圖
                moved_in_this_pass = True
                break
            elif can_move_up(r, c, im):
                move_up(r, c)         # 執行向上移動
                im = pyg.screenshot() # 移動後重新截圖
                moved_in_this_pass = True
                break
            elif can_move_down(r, c, im):
                move_down(r, c)       # 執行向下移動
                im = pyg.screenshot() # 移動後重新截圖
                moved_in_this_pass = True
                break
        
        # 若找到並執行了一次可移動操作，跳出內層迴圈，讓主迴圈重新從最新截圖分析
        if moved_in_this_pass:
            break

    # 避免迴圈過度佔用 CPU，可加上極短微秒等級的暫停
    time.sleep(0.05)

開始自動遊戲機制...
